In [21]:
import pandas as pd
import numpy as np
import pickle
import time

from sklearn.linear_model import LogisticRegressionCV, RidgeClassifierCV
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

In [22]:
models_to_validate = [
    LogisticRegressionCV,
    RidgeClassifierCV,
    RandomForestClassifier,
    CatBoostClassifier,
    LGBMClassifier,
    XGBClassifier,
    MultinomialNB
]

In [25]:
fitted_models = {}
for model in models_to_validate:
    with open(f'model_best_weights/{model.__name__}.pickle', 'rb') as handle:
        fitted_models[model.__name__] = pickle.load(handle)

/var/folders/cl/r93zvffj0sz_6qfhc44pj6mr0000gn/T/ipykernel_29911/2155067038.py:4: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  fitted_models[model.__name__] = pickle.load(handle)
/Users/sidorovegor/Desktop/projects/python/misis/MISIS_sentiment_analysis_project/.venv/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.5.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For 

In [26]:
val_df = pd.read_csv('DataBases/external_testing_dataset.csv')
val_df

,message,sentiment,sentiment_label
0,Thank you for the amazing attention.,1,positive
1,I appreciated how you handled the situation. T...,1,positive
2,I've been reflecting on the service I received...,1,positive
3,I wanted to thank you for the pleasant service...,1,positive
4,My experience with this service was completely...,0,negative
...,...,...,...
995,"After experiencing the response from the team,...",0,negative
996,"In today's competitive market, the somewhat me...",0,negative
997,Nice work! Five stars!,1,positive
998,Your team provided somewhat unsatisfactory cus...,0,negative


In [27]:
messages = val_df['message']
labels = val_df['sentiment']

In [28]:
def show_report(models, x_test, y_test):
    df = pd.DataFrame()
    for model_name, model in models.items():
        start_time = time.time()
        pred = model.predict(x_test)
        predict_time = time.time() - start_time
        metrics = {
            'precision': precision_score(y_test, pred),
            'recall': recall_score(y_test, pred),
            'f1': f1_score(y_test, pred),
            'roc_auc': roc_auc_score(y_test, pred),
            'predict_time_sec': predict_time}
        df[model_name] = metrics
    return df

In [29]:
rep = show_report(fitted_models, messages, labels)
rep

/Users/sidorovegor/Desktop/projects/python/misis/MISIS_sentiment_analysis_project/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,LogisticRegressionCV,RidgeClassifierCV,RandomForestClassifier,CatBoostClassifier,LGBMClassifier,XGBClassifier,MultinomialNB
precision,0.864119,0.801314,0.504541,0.826667,0.823779,0.850746,0.664894
recall,0.814000,0.976000,1.000000,0.744000,0.776000,0.798000,1.000000
f1,0.838311,0.880072,0.670691,0.783158,0.799176,0.823529,0.798722
roc_auc,0.843000,0.867000,0.509000,0.794000,0.805000,0.829000,0.748000
predict_time_sec,0.010976,0.008338,0.012368,0.021440,0.177378,0.022953,0.004543


In [30]:
val_df.drop(['sentiment_label'], axis=1, inplace=True)

In [31]:
for model_name, model in fitted_models.items():
    y_pred = model.predict(messages)
    val_df[f'{model_name}_predict'] = y_pred

    if model_name != 'RidgeClassifierCV':
        score = np.max(model.predict_proba(messages), axis=1)
        val_df[f'{model_name}_proba'] = score

/Users/sidorovegor/Desktop/projects/python/misis/MISIS_sentiment_analysis_project/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/sidorovegor/Desktop/projects/python/misis/MISIS_sentiment_analysis_project/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [35]:
val_df.iloc[:15, :7]

,message,sentiment,LogisticRegressionCV_predict,LogisticRegressionCV_proba,RidgeClassifierCV_predict,RandomForestClassifier_predict,RandomForestClassifier_proba
0,Thank you for the amazing attention.,1,1,0.995108,1,1,0.545113
1,I appreciated how you handled the situation. T...,1,1,0.660883,1,1,0.534526
2,I've been reflecting on the service I received...,1,1,0.856431,1,1,0.548868
3,I wanted to thank you for the pleasant service...,1,1,0.998080,1,1,0.546792
4,My experience with this service was completely...,0,0,0.987000,0,1,0.515060
5,I'm writing to express my satisfaction with th...,1,1,0.501732,1,1,0.533400
6,After experiencing the assistance from this se...,1,0,0.505430,1,1,0.529364
7,I valued the service.,1,1,0.999997,1,1,0.546399
8,Your team provided extremely exceptional assis...,1,1,0.621197,1,1,0.534498
9,I'm writing to express my dissatisfaction with...,0,0,0.737259,0,1,0.533400


In [37]:
val_df.shape

(1000, 15)

In [39]:
val_df.iloc[:15, [0, 1, 7, 8, 9, 10, 11, 12]]

,message,sentiment,CatBoostClassifier_predict,CatBoostClassifier_proba,LGBMClassifier_predict,LGBMClassifier_proba,XGBClassifier_predict,XGBClassifier_proba
0,Thank you for the amazing attention.,1,1,0.939844,1,0.969603,1,0.995975
1,I appreciated how you handled the situation. T...,1,0,0.588265,0,0.644074,0,0.675691
2,I've been reflecting on the service I received...,1,1,0.885125,1,0.955621,1,0.978581
3,I wanted to thank you for the pleasant service...,1,1,0.977453,1,0.989312,1,0.999630
4,My experience with this service was completely...,0,0,0.714971,0,0.893561,0,0.922366
5,I'm writing to express my satisfaction with th...,1,0,0.588265,1,0.670584,1,0.524194
6,After experiencing the assistance from this se...,1,1,0.599272,1,0.605236,1,0.666021
7,I valued the service.,1,1,0.929039,1,0.983042,1,0.993551
8,Your team provided extremely exceptional assis...,1,1,0.577294,1,0.669602,1,0.725973
9,I'm writing to express my dissatisfaction with...,0,0,0.588265,0,0.615880,0,0.614232


In [41]:
val_df.iloc[:15, [0, 1, 13, 14]]

,message,sentiment,MultinomialNB_predict,MultinomialNB_proba
0,Thank you for the amazing attention.,1,1,0.827256
1,I appreciated how you handled the situation. T...,1,1,0.800366
2,I've been reflecting on the service I received...,1,1,0.890984
3,I wanted to thank you for the pleasant service...,1,1,0.869301
4,My experience with this service was completely...,0,0,0.774382
5,I'm writing to express my satisfaction with th...,1,1,0.766629
6,After experiencing the assistance from this se...,1,1,0.688724
7,I valued the service.,1,1,0.955140
8,Your team provided extremely exceptional assis...,1,1,0.785335
9,I'm writing to express my dissatisfaction with...,0,0,0.508358
